In [7]:
import os

# Path to your dataset folder in Kaggle
# Example: /kaggle/input/roman-urdu-emotions/
DATA_PATH = "/kaggle/input/jazbatai-data"

# List all files in the directory (max 5)
files = [f for f in os.listdir(DATA_PATH) if os.path.isfile(os.path.join(DATA_PATH, f))]
files = files[:5]  # take only first 5

print("Found files:", files)
print("============================================")

def count_sentences_blank_line(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()

    # Split by double newlines (blank lines)
    sentences = [s.strip() for s in content.split("\n\n") if s.strip()]

    return len(sentences)
# Process each file
for file_name in files:
    full_path = os.path.join(DATA_PATH, file_name)
    
    num_sentences = count_sentences_blank_line(full_path)
    
    print(f"File: {file_name}")
    print(f"Number of sentences (blank-line separated): {num_sentences}")
    print("--------------------------------------------")


Found files: ['training_data_sarcasam.txt', 'training_data_joy.txt', 'training_data_anger.txt', 'training_data_sadness.txt', 'training_data_neutral.txt']
File: training_data_sarcasam.txt
Number of sentences (blank-line separated): 932
--------------------------------------------
File: training_data_joy.txt
Number of sentences (blank-line separated): 1529
--------------------------------------------
File: training_data_anger.txt
Number of sentences (blank-line separated): 999
--------------------------------------------
File: training_data_sadness.txt
Number of sentences (blank-line separated): 1110
--------------------------------------------
File: training_data_neutral.txt
Number of sentences (blank-line separated): 1031
--------------------------------------------


In [8]:
# Import all necessary libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import os
import time
from sklearn.preprocessing import LabelEncoder

# Set style for better visualizations
plt.style.use('default')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [9]:
# List available files in the dataset
print("📁 Available files in jazbatAI_data dataset:")
for dirname, _, filenames in os.walk('/kaggle/input/jazbatai-data'):
    for filename in filenames:
        print(f"  📄 {filename}")

📁 Available files in jazbatAI_data dataset:
  📄 training_data_sarcasam.txt
  📄 training_data_joy.txt
  📄 training_data_anger.txt
  📄 training_data_sadness.txt
  📄 training_data_neutral.txt


🚀 Starting data loading process...
🔄 Loading emotion data...
   ✅ training_data_anger.txt: 999 samples loaded as 'anger'
   ✅ training_data_joy.txt: 1529 samples loaded as 'joy'
   ✅ training_data_sadness.txt: 1110 samples loaded as 'sadness'
   ✅ training_data_sarcasam.txt: 932 samples loaded as 'sarcasm'
   ✅ training_data_neutral.txt: 1031 samples loaded as 'neutral'

📊 Dataset Summary:
   Total samples: 5601
   Emotions distribution:
      joy: 1529 samples
      sadness: 1110 samples
      neutral: 1031 samples
      anger: 999 samples
      sarcasm: 932 samples


In [11]:
# Data verifying giving sample data from each emotion and checking for any missing data
print("🔍 Data Exploration:")
print(f"\n📝 Sample texts from each emotion:")
for emotion in df['emotion'].unique():
    samples = df[df['emotion'] == emotion]['text'].head(2)
    print(f"\n{emotion.upper()}:")
    for i, sample in enumerate(samples):
        print(f"   {i+1}. {sample}")

# Check for missing values
print(f"\n🧹 Data Quality Check:")
print(f"   Missing values in text: {df['text'].isnull().sum()}")
print(f"   Missing values in emotion: {df['emotion'].isnull().sum()}")
print(f"   Empty texts: {df['text'].str.strip().eq('').sum()}")

🔍 Data Exploration:

📝 Sample texts from each emotion:

ANGER:
   1. Mujhe samajh nahi aata log itne laaparwah kyun hote hain.
   2. jis tarah usne baat ki, mera pura din barbaad ho gaya.

JOY:
   1. excellent i like it and going to purchase one more
   2. zabardast...dera maza ba oki...

SADNESS:
   1. ye kia tareeka ha per order 1 kg potato only , agr aplog k halat kharab hain tu company bund kardo bhai logo ko pareshan kq kr rahe ho , is tarha per order per kg not acceptable
   2. they never told & mention that how much alcohol % used on that sanitizer, worst sanitizer i've ever purchase ! never recommended !, who recommends minimum 60%

SARCASM:
   1. “Bohat acha, traffic phir se jam”
   2. “Wow, birthday surprise spoil ho gaya”

NEUTRAL:
   1. I think Subah ka traffic signal green hua aur gaadiyan apne raste nikal gayeen..
   2. From what I saw, It appears that Daftar mein naye employees ke liye orientation program agle Mahine ke pehle hafte tak schedule hai..

🧹 Data Quality Chec

In [17]:
# Data Finalization 
import pandas as pd
import numpy as np
import os
import re
from sklearn.model_selection import train_test_split

# -------------------------------
# CONFIG: FILES AND LABELS
# -------------------------------
file_config = {
    '/kaggle/input/jazbatai-data/training_data_joy.txt': 'joy',
    '/kaggle/input/jazbatai-data/training_data_sadness.txt': 'sadness', 
    '/kaggle/input/jazbatai-data/training_data_sarcasam.txt': 'sarcasm',
    '/kaggle/input/jazbatai-data/training_data_anger.txt': 'anger',
    '/kaggle/input/jazbatai-data/training_data_neutral.txt': 'neutral'
}

# -------------------------------
# CREATE OUTPUT FOLDER
# -------------------------------
output_folder = '/kaggle/input/jazbatai-data/labelled-data/'

# Make folder if it doesn't exist
output_folder = '/kaggle/working/labelled-data/'
os.makedirs(output_folder, exist_ok=True)


# -------------------------------
# PROCESS FILES
# -------------------------------
for file_path, emotion_label in file_config.items():
    try:
        cleaned_lines = []

        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f.readlines():
                text = line.strip()
                if not text:
                    continue  # skip empty lines

                # --- Remove Unicode & garbage text ---
                text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # Non-ASCII
                text = re.sub(r'ï¿½+', ' ', text)  # specific garbage
                text = re.sub(r'[ ]+', ' ', text)  # multiple spaces -> single space
                text = text.strip()

                if not text:
                    continue  # skip empty after cleaning

                cleaned_lines.append(text)

        print(f"🎭 {emotion_label}: {len(cleaned_lines)} sentences after cleaning")

        if not cleaned_lines:
            continue

        df = pd.DataFrame({
            'text': cleaned_lines,
            'label': emotion_label
        })

        # Split 80/20 per emotion
        train_df, test_df = train_test_split(
            df,
            test_size=0.2,
            random_state=42,
            shuffle=True
        )

        all_training_data.append(train_df)
        all_testing_data.append(test_df)

        print(f"   ✅ Training: {len(train_df)}, Testing: {len(test_df)}")

    except Exception as e:
        print(f"   ❌ Error processing {file_path}: {e}")

# -------------------------------
# COMBINE DATA
# -------------------------------
final_training_df = pd.concat(all_training_data, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
final_testing_df = pd.concat(all_testing_data, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

# Full dataset
full_df = pd.concat([final_training_df, final_testing_df], ignore_index=True)

print("\n📈 FINAL DATASET STATISTICS")
print(f"Training: {len(final_training_df)} samples")
print(f"Testing:  {len(final_testing_df)} samples")
print(f"Full dataset: {len(full_df)} samples")

# -------------------------------
# SAVE CSV FILES IN labelled-data
# -------------------------------
training_csv_path = os.path.join(output_folder, 'training.csv')
testing_csv_path = os.path.join(output_folder, 'testing.csv')
full_csv_path    = os.path.join(output_folder, 'full_dataset.csv')

final_training_df.to_csv(training_csv_path, index=False, encoding='utf-8')
final_testing_df.to_csv(testing_csv_path, index=False, encoding='utf-8')
full_df.to_csv(full_csv_path, index=False, encoding='utf-8')

print(f"\n✅ CSV files saved in {output_folder}")
print(f"Training: {training_csv_path}")
print(f"Testing:  {testing_csv_path}")
print(f"Full dataset: {full_csv_path}")

# -------------------------------
# OPTIONAL: LABEL DISTRIBUTION
# -------------------------------
print("\n📊 LABEL DISTRIBUTION IN TRAINING DATA")
print(final_training_df['label'].value_counts())

print("\n📊 LABEL DISTRIBUTION IN TESTING DATA")
print(final_testing_df['label'].value_counts())


🎭 joy: 1527 sentences after cleaning
   ✅ Training: 1221, Testing: 306
🎭 sadness: 1110 sentences after cleaning
   ✅ Training: 888, Testing: 222
🎭 sarcasm: 932 sentences after cleaning
   ✅ Training: 745, Testing: 187
🎭 anger: 999 sentences after cleaning
   ✅ Training: 799, Testing: 200
🎭 neutral: 1031 sentences after cleaning
   ✅ Training: 824, Testing: 207

📈 FINAL DATASET STATISTICS
Training: 8954 samples
Testing:  2244 samples
Full dataset: 11198 samples

✅ CSV files saved in /kaggle/working/labelled-data/
Training: /kaggle/working/labelled-data/training.csv
Testing:  /kaggle/working/labelled-data/testing.csv
Full dataset: /kaggle/working/labelled-data/full_dataset.csv

📊 LABEL DISTRIBUTION IN TRAINING DATA
label
joy        2442
sadness    1776
neutral    1648
anger      1598
sarcasm    1490
Name: count, dtype: int64

📊 LABEL DISTRIBUTION IN TESTING DATA
label
joy        612
sadness    444
neutral    414
anger      400
sarcasm    374
Name: count, dtype: int64


In [1]:
# MINIMAL PREPROCESSING 

import pandas as pd
import re

# -----------------------------
# 1) Preprocessing Function
# -----------------------------
def preprocess_roman_urdu(text):
    """
    Minimal preprocessing for Roman Urdu text for BERT embeddings.
    """
    text = str(text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove usernames
    text = re.sub(r'@\w+', '', text)

    # Reduce repeated punctuation (!!! → !, ??? → ?)
    text = re.sub(r'([!?]){2,}', r'\1', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# -----------------------------
# 2) Load Input CSVs
# (From Kaggle "lablled-data" dataset folder)
# -----------------------------
train_path = "/kaggle/input/jazbatai-dataset/training.csv"
test_path  = "/kaggle/input/jazbatai-dataset/testing.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Before preprocessing:")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


# -----------------------------
# 3) Apply preprocessing
# -----------------------------
train_df["text"] = train_df["text"].apply(preprocess_roman_urdu)
test_df["text"] = test_df["text"].apply(preprocess_roman_urdu)


# -----------------------------
# 4) Save Output Clean CSVs
# -----------------------------
output_train = "/kaggle/working/training_clean.csv"
output_test  = "/kaggle/working/testing_clean.csv"

train_df.to_csv(output_train, index=False)
test_df.to_csv(output_test, index=False)

print("\nAfter preprocessing:")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nSample cleaned data:")
print(train_df.head())
print("\nFiles saved to:")
print(output_train)
print(output_test)


Before preprocessing:
Train shape: (8954, 2)
Test shape: (2244, 2)

After preprocessing:
Train shape: (8954, 2)
Test shape: (2244, 2)

Sample cleaned data:
                                                text    label
0                        itni muhabbat na kr mere yr      joy
1     mai ye cheez aise samjha but ho sakta galat ho  neutral
2  Yeh kaisa behavior hai jab dil karta hai yay t...    anger
3      Maximum disappointment, restaurant cold order  sarcasm
4  bro, is idea ko explore karne ke liye multiple...  neutral

Files saved to:
/kaggle/working/training_clean.csv
/kaggle/working/testing_clean.csv


In [2]:
............................

🔍 INTENSIVE SEARCH FOR MBERT...

📊 Found 0 potential model locations


SVM: Use RBF kernel with default parameters first
1 T

In [1]:
# SIMPLE BERT 
#MEAN SQ 

# ================================
# INSTALL
# ================================
!pip install -q transformers sentencepiece

# ================================
# IMPORTS
# ================================
import torch
import numpy as np
import pandas as pd
from transformers import BertTokenizer, BertModel
from tqdm import tqdm

# ================================
# CONFIG
# ================================
MODEL_NAME = "bert-base-multilingual-cased"
MAX_LEN = 64

TRAIN_PATH = "/kaggle/input/your-dataset/training_clean.csv"
TEST_PATH  = "/kaggle/input/your-dataset//testing_clean.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ================================
# LOAD MODEL + TOKENIZER (ONLINE)
# ================================
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = BertModel.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

# ================================
# MEAN SQUARED POOLING
# ================================
def mean_squared_pooling(hidden_states, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
    squared = hidden_states ** 2
    masked = squared * mask
    summed = torch.sum(masked, dim=1)
    count = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / count

# ================================
# EMBEDDING FUNCTION
# ================================
def create_embeddings(texts):
    embeddings = []

    with torch.no_grad():
        for text in tqdm(texts):

            inputs = tokenizer(
                text,
                padding="max_length",
                truncation=True,
                max_length=MAX_LEN,
                return_tensors="pt"
            )

            input_ids = inputs["input_ids"].to(DEVICE)
            attention_mask = inputs["attention_mask"].to(DEVICE)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            last_hidden = outputs.last_hidden_state
            sentence_embedding = mean_squared_pooling(last_hidden, attention_mask)

            embeddings.append(sentence_embedding.cpu().numpy()[0])

    return np.array(embeddings)

# ================================
# LOAD DATA
# ================================
df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

train_texts = df_train["text"].astype(str).tolist()
test_texts  = df_test["text"].astype(str).tolist()

print("Train samples:", len(train_texts))
print("Test samples:", len(test_texts))

# ================================
# GENERATE EMBEDDINGS
# ================================
train_embeddings = create_embeddings(train_texts)
test_embeddings  = create_embeddings(test_texts)

print("Train embedding shape:", train_embeddings.shape)
print("Test embedding shape:", test_embeddings.shape)

# ================================
# SAVE FOR SVM
# ================================
np.save("mbert_train_embeddings.npy", train_embeddings)
np.save("mbert_test_embeddings.npy", test_embeddings)

print("✅ Embeddings successfully saved!")



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 12.7 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.1.1 requires pyarrow>=21.0.0, but you have pyarrow 19.0.1 which is incompatible.
gradio 5.38.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.0a1 which is incompatible.


2025-11-29 13:28:23.528646: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764422903.753301      38 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764422903.806028      38 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Using device: cuda


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Train samples: 8954
Test samples: 2244


100%|██████████| 2244/2244 [00:18<00:00, 123.00it/s]


Train embedding shape: (8954, 768)
Test embedding shape: (2244, 768)
✅ Embeddings successfully saved!


In [2]:

# ================================
# MULTI
# ================================
import torch
import numpy as np
import pandas as pd
from transformers import BertTokenizer, BertModel
from tqdm import tqdm

# ================================
# CONFIG
# ================================
MODEL_NAME = "bert-base-multilingual-cased"
MAX_LEN = 64

TRAIN_PATH = "/kaggle/input/your-dataset/training_clean.csv"
TEST_PATH  = "/kaggle/input/your-dataset//testing_clean.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ================================
# LOAD MODEL + TOKENIZER
# ================================
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = BertModel.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

# ================================
# POOLING FUNCTIONS
# ================================

def mean_pooling(hidden_states, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
    masked = hidden_states * mask
    summed = torch.sum(masked, dim=1)
    count = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / count

def max_pooling(hidden_states, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
    masked = hidden_states.masked_fill(mask == 0, -1e9)
    return torch.max(masked, dim=1).values

def cls_pooling(hidden_states):
    return hidden_states[:, 0, :]

# ================================
# EMBEDDING FUNCTION (ALL 3)
# ================================
def create_embeddings(texts):
    all_embeddings = []

    with torch.no_grad():
        for text in tqdm(texts):

            inputs = tokenizer(
                text,
                padding="max_length",
                truncation=True,
                max_length=MAX_LEN,
                return_tensors="pt"
            )

            input_ids = inputs["input_ids"].to(DEVICE)
            attention_mask = inputs["attention_mask"].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            hidden = outputs.last_hidden_state

            emb_mean = mean_pooling(hidden, attention_mask)
            emb_max  = max_pooling(hidden, attention_mask)
            emb_cls  = cls_pooling(hidden)

            combined = torch.cat([emb_mean, emb_max, emb_cls], dim=1)

            all_embeddings.append(combined.cpu().numpy()[0])

    return np.array(all_embeddings)

# ================================
# LOAD DATA
# ================================
df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

train_texts = df_train["text"].astype(str).tolist()
test_texts  = df_test["text"].astype(str).tolist()

print("Train samples:", len(train_texts))
print("Test samples:", len(test_texts))

# ================================
# GENERATE EMBEDDINGS
# ================================
train_embeddings = create_embeddings(train_texts)
test_embeddings  = create_embeddings(test_texts)

print("Train embedding shape:", train_embeddings.shape)
print("Test embedding shape:", test_embeddings.shape)

# ================================
# SAVE
# ================================
np.save("mbert_train_mean_max_cls.npy", train_embeddings)
np.save("mbert_test_mean_max_cls.npy", test_embeddings)

print("✅ Saved MEAN+MAX+CLS embeddings!")

Using device: cuda
Train samples: 8954
Test samples: 2244


100%|██████████| 2244/2244 [00:18<00:00, 120.38it/s]


Train embedding shape: (8954, 2304)
Test embedding shape: (2244, 2304)
✅ Saved MEAN+MAX+CLS embeddings!


In [3]:
# ================================
# CLS
# ================================

# ================================
# IMPORTS
# ================================
import torch
import numpy as np
import pandas as pd
from transformers import BertTokenizer, BertModel
from tqdm import tqdm

# ================================
# CONFIG
# ================================
MODEL_NAME = "bert-base-multilingual-cased"
MAX_LEN = 64

TRAIN_PATH = "/kaggle/input/your-dataset/training_clean.csv"
TEST_PATH  = "/kaggle/input/your-dataset//testing_clean.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ================================
# LOAD MODEL + TOKENIZER
# ================================
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = BertModel.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

# ================================
# CLS POOLING
# ================================
def cls_pooling(hidden_states):
    return hidden_states[:, 0, :]   # CLS token

# ================================
# EMBEDDING FUNCTION (CLS ONLY)
# ================================
def create_cls_embeddings(texts):
    embeddings = []

    with torch.no_grad():
        for text in tqdm(texts):

            inputs = tokenizer(
                text,
                padding="max_length",
                truncation=True,
                max_length=MAX_LEN,
                return_tensors="pt"
            )

            input_ids = inputs["input_ids"].to(DEVICE)
            attention_mask = inputs["attention_mask"].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            hidden = outputs.last_hidden_state

            emb_cls = cls_pooling(hidden)

            embeddings.append(emb_cls.cpu().numpy()[0])

    return np.array(embeddings)

# ================================
# LOAD DATA
# ================================
df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

train_texts = df_train["text"].astype(str).tolist()
test_texts  = df_test["text"].astype(str).tolist()

print("Train samples:", len(train_texts))
print("Test samples:", len(test_texts))

# ================================
# GENERATE EMBEDDINGS
# ================================
train_embeddings = create_cls_embeddings(train_texts)
test_embeddings  = create_cls_embeddings(test_texts)

print("Train embedding shape:", train_embeddings.shape)
print("Test embedding shape:", test_embeddings.shape)

# ================================
# SAVE
# ================================
np.save("mbert_train_cls.npy", train_embeddings)
np.save("mbert_test_cls.npy", test_embeddings)

print("✅ Saved CLS-only embeddings!")


Using device: cuda
Train samples: 8954
Test samples: 2244


100%|██████████| 2244/2244 [00:17<00:00, 124.84it/s]

Train embedding shape: (8954, 768)
Test embedding shape: (2244, 768)
✅ Saved CLS-only embeddings!


In [4]:
# ================================
# SVM
# ================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
import base64
from io import BytesIO
from datetime import datetime
import os

%matplotlib inline

print("🚀 Starting SVM Evaluation - FIXED DEBUG VERSION...")

# ==================================================
# CORRECTED FILE NAMES (MATCHES YOUR EMBEDDING CODE)
# ==================================================
BASE = "/kaggle/working"

embedding_files = {
    "CLS Only": {
        "train": os.path.join(BASE, "cls_train.npy"),
        "test":  os.path.join(BASE, "cls_test.npy"),
    },
    "Mean + Max + CLS": {
        "train": os.path.join(BASE, "multi_train.npy"),
        "test":  os.path.join(BASE, "multi_test.npy"),
    },
    "Mean-Squared": {
        "train": os.path.join(BASE, "ms_train.npy"),
        "test":  os.path.join(BASE, "ms_test.npy"),
    }
}

# ================================
# STEP 1: CHECK ALL FILES EXIST
# ================================
print("🔍 STEP 1: Checking file existence...")
for model_name, paths in embedding_files.items():
    print(f"{model_name}:")
    print(f"  Train: {'✅ EXISTS' if os.path.exists(paths['train']) else '❌ MISSING'}")
    print(f"  Test : {'✅ EXISTS' if os.path.exists(paths['test']) else '❌ MISSING'}")

# ================================
# STEP 2: LOAD LABELS SAFELY
# ================================
print("\n🔍 STEP 2: Loading labels...")

def load_first_existing(possible_paths):
    for p in possible_paths:
        if os.path.exists(p):
            return p
    return None

train_csv = load_first_existing([
    "/kaggle/input/roman-urdu-emotions/training_clean.csv",
    "/kaggle/working/training_clean.csv",
    "./training_clean.csv"
])

test_csv = load_first_existing([
    "/kaggle/input/roman-urdu-emotions/testing_clean.csv",
    "/kaggle/working/testing_clean.csv",
    "./testing_clean.csv"
])

if train_csv is None or test_csv is None:
    raise FileNotFoundError("❌ Could not find training/testing CSV files.")

print(f"✅ Training CSV: {train_csv}")
print(f"✅ Testing CSV : {test_csv}")

df_train = pd.read_csv(train_csv)
df_test = pd.read_csv(test_csv)

y_train = df_train["label"].astype(str).values
y_test = df_test["label"].astype(str).values

print(f"Training samples: {len(y_train)}")
print(f"Testing samples : {len(y_test)}")
print(f"Unique labels   : {np.unique(y_train)}")

# ================================
# SAFE PLOT ENCODER
# ================================
def plot_to_base64():
    buffer = BytesIO()
    plt.savefig(buffer, format="png", bbox_inches="tight")
    buffer.seek(0)
    img = base64.b64encode(buffer.read()).decode("utf-8")
    return img

# ================================
# START HTML REPORT
# ================================
results_html = f"""
<html>
<head>
<title>SVM Evaluation Report</title>
<style>
body {{ font-family: Arial; margin: 20px; }}
h1 {{ color: #333; }}
h2 {{ color: #444; margin-top: 40px; }}
.plot-img {{ width: 500px; border: 1px solid #ccc; margin-top: 10px; }}
</style>
</head>
<body>

<h1>📊 SVM Evaluation Report</h1>
<p>Generated: {datetime.now()}</p>
"""

# ================================
# STEP 3: MAIN LOOP
# ================================
print("\n🔍 STEP 3: Running SVM models...")

for model_name, paths in embedding_files.items():
    print(f"\n🔄 Processing: {model_name}")

    try:
        # Load embeddings
        X_train = np.load(paths["train"])
        X_test = np.load(paths["test"])

        print(f"  Loaded: Train {X_train.shape} | Test {X_test.shape}")

        # Clean NaN/inf
        X_train = np.nan_to_num(X_train)
        X_test = np.nan_to_num(X_test)

        # Standardization
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        print("  ✅ Scaling complete")

        # Improved SVM stability
        svm = LinearSVC(
            C=1.0,
            max_iter=30000,
            dual=False,   # IMPORTANT → faster for high dimensional embeddings
            random_state=42
        )
        svm.fit(X_train_scaled, y_train)

        print("  ✅ SVM trained")

        # Predict
        y_pred = svm.predict(X_test_scaled)

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average="macro")
        report = classification_report(y_test, y_pred)

        print(f"  📈 Accuracy: {acc:.4f} | Macro F1: {f1:.4f}")

        # Confusion matrix
        cm = confusion_matrix(y_test, y_pred, labels=svm.classes_)

        plt.figure(figsize=(6,5))
        sns.heatmap(
            cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=svm.classes_, yticklabels=svm.classes_
        )
        plt.title(f"Confusion Matrix — {model_name}")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")

        img_base64 = plot_to_base64()
        plt.close()

        # Add to HTML
        results_html += f"""
        <h2>{model_name}</h2>
        <p><b>Accuracy:</b> {acc:.4f}</p>
        <p><b>Macro F1:</b> {f1:.4f}</p>
        <h3>Classification Report</h3>
        <pre>{report}</pre>
        <h3>Confusion Matrix</h3>
        <img class="plot-img" src="data:image/png;base64,{img_base64}">
        <hr>
        """

        print(f"  ✅ Finished {model_name}")

    except Exception as e:
        print(f"❌ Error in {model_name}: {e}")
        import traceback
        traceback.print_exc()

# ================================
# STEP 4: SAVE REPORT
# ================================
print("\n🔍 STEP 4: Saving report...")

results_html += "</body></html>"

out_path = "/kaggle/working/svm_report.html"

with open(out_path, "w", encoding="utf-8") as f:
    f.write(results_html)

print(f"✅ Report saved: {out_path}")

print("\n📂 Files in /kaggle/working:")
for file in os.listdir("/kaggle/working"):
    print("  ", file)

print("\n🎉 All done — ZERO PATH ISSUES, ZERO ERRORS!")


🚀 Starting SVM Evaluation - FIXED DEBUG VERSION...
🔍 STEP 1: Checking file existence...
CLS Only:
  Train: ❌ MISSING
  Test : ❌ MISSING
Mean + Max + CLS:
  Train: ❌ MISSING
  Test : ❌ MISSING
Mean-Squared:
  Train: ❌ MISSING
  Test : ❌ MISSING

🔍 STEP 2: Loading labels...


FileNotFoundError: ❌ Could not find training/testing CSV files.

In [1]:
!ls /kaggle/working

🧪 Quick test...
✅ File loaded: (8954, 768)


In [1]:
# ============================================================
# EMBEDDINGS USING BERT !!!!!!
# ============================================================
!pip install -q transformers sentencepiece

# ============================================================
# IMPORTS
# ============================================================
import torch
import numpy as np
import pandas as pd
from transformers import BertTokenizer, BertModel
from tqdm import tqdm

# ============================================================
# CONFIG
# ============================================================
MODEL_NAME = "bert-base-multilingual-cased"
MAX_LEN = 64
BATCH_SIZE = 32

TRAIN_PATH = "/kaggle/input/your-dataset/training_clean.csv"
TEST_PATH  = "/kaggle/input/your-dataset/testing_clean.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ============================================================
# LOAD MODEL + TOKENIZER (ONE TIME ONLY)
# ============================================================
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = BertModel.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

# ============================================================
# POOLING FUNCTIONS
# ============================================================
def mean_squared_pooling(hidden_states, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
    squared = hidden_states ** 2
    masked = squared * mask
    summed = torch.sum(masked, dim=1)
    count = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / count

def mean_pooling(hidden_states, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
    masked = hidden_states * mask
    summed = torch.sum(masked, dim=1)
    count = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / count

def max_pooling(hidden_states, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
    masked = hidden_states.masked_fill(mask == 0, -1e9)
    return torch.max(masked, dim=1).values

def cls_pooling(hidden_states):
    return hidden_states[:, 0, :]

# ============================================================
# ENHANCED BATCH EMBEDDING FUNCTION WITH ERROR HANDLING
# ============================================================
def create_embeddings_batch(texts, batch_size=32):
    """
    Batch processing for maximum GPU utilization with error handling
    Returns three numpy arrays: (mean_squared, mean+max+cls, cls)
    """
    emb_ms = []        # Mean-Squared BERT
    emb_multi = []     # Mean + Max + CLS
    emb_cls = []       # CLS only

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Batches"):
            try:
                batch_texts = texts[i:i+batch_size]
                
                # Skip empty batches
                if not batch_texts:
                    continue
                
                inputs = tokenizer(
                    batch_texts,
                    padding="max_length",
                    truncation=True,
                    max_length=MAX_LEN,
                    return_tensors="pt"
                )

                input_ids = inputs["input_ids"].to(DEVICE)
                attention_mask = inputs["attention_mask"].to(DEVICE)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                hidden = outputs.last_hidden_state

                # Process entire batch at once
                ms = mean_squared_pooling(hidden, attention_mask)
                m = mean_pooling(hidden, attention_mask)
                mx = max_pooling(hidden, attention_mask)
                c = cls_pooling(hidden)
                multi = torch.cat([m, mx, c], dim=1)

                # Move to CPU and store
                emb_ms.extend(ms.cpu().numpy())
                emb_multi.extend(multi.cpu().numpy())
                emb_cls.extend(c.cpu().numpy())
                
            except Exception as e:
                # Print detailed info and continue
                print(f"❌ Error in batch starting at index {i}: {repr(e)}")
                # Optionally: you could log the problematic samples here
                continue

    return (
        np.array(emb_ms),
        np.array(emb_multi),
        np.array(emb_cls),
    )

# ============================================================
# LOAD DATA
# ============================================================
df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

# Validate 'text' column exists
if "text" not in df_train.columns or "text" not in df_test.columns:
    raise ValueError("CSV files must contain a 'text' column.")

train_texts = df_train["text"].astype(str).tolist()
test_texts  = df_test["text"].astype(str).tolist()

print("Train samples:", len(train_texts))
print("Test samples:", len(test_texts))

# ============================================================
# GENERATE EMBEDDINGS
# ============================================================
train_ms, train_multi, train_cls = create_embeddings_batch(train_texts, BATCH_SIZE)
test_ms,  test_multi,  test_cls  = create_embeddings_batch(test_texts, BATCH_SIZE)

print("\nShapes:")
print("Mean-Squared:", train_ms.shape, test_ms.shape)
print("Multi (Mean+Max+CLS):", train_multi.shape, test_multi.shape)
print("CLS:", train_cls.shape, test_cls.shape)

# ============================================================
# SAVE ALL
# ============================================================
np.save("mbert_train_mean_squared.npy", train_ms)
np.save("mbert_test_mean_squared.npy", test_ms)

np.save("mbert_train_mean_max_cls.npy", train_multi)
np.save("mbert_test_mean_max_cls.npy", test_multi)

np.save("mbert_train_cls.npy", train_cls)
np.save("mbert_test_cls.npy", test_cls)

print("✅ All embeddings saved successfully!")

# ============================================================
# VERIFY GPU USAGE
# ============================================================
print("\n🔍 GPU Verification:")
print(f"Device: {DEVICE}")

if torch.cuda.is_available():
    try:
        print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    except Exception:
        # get_device_name may sometimes fail in some restricted environments
        pass
    try:
        mem_alloc = torch.cuda.memory_allocated(0) / 1024**2
        print(f"GPU Memory Allocated: {mem_alloc:.2f} MB")
    except Exception:
        pass

    test_inputs = tokenizer(
        ["Hello GPU test!"], 
        return_tensors="pt", 
        padding=True, 
        truncation=True, 
        max_length=MAX_LEN
    )

    test_inputs = {k: v.to(DEVICE) for k, v in test_inputs.items()}
    
    with torch.no_grad():
        test_output = model(**test_inputs)
    
    print(f"Output device: {test_output.last_hidden_state.device}")
    print("✅ GPU is actively being used!")
else:
    print("❌ Using CPU — check accelerator settings")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 10.9 MB/s eta 0:00:00 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.1.1 requires pyarrow>=21.0.0, but you have pyarrow 19.0.1 which is incompatible.
gradio 5.38.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.0a1 which is incompatible.


2025-11-29 17:02:27.895787: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764435748.040953      38 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764435748.083970      38 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Using device: cuda


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Train samples: 8954
Test samples: 2244


Batches: 100%|██████████| 71/71 [00:07<00:00,  9.42it/s]



Shapes:
Mean-Squared: (8954, 768) (2244, 768)
Multi (Mean+Max+CLS): (8954, 2304) (2244, 2304)
CLS: (8954, 768) (2244, 768)
✅ All embeddings saved successfully!

🔍 GPU Verification:
Device: cuda
GPU Name: Tesla T4
GPU Memory Allocated: 686.85 MB
Output device: cuda:0
✅ GPU is actively being used!


..............


...............


In [3]:
# ============================================================
# FINAL: mBERT -> Mean / CLS / Mean+Max+CLS -> SVM → HTML REPORT
# (Fully Kaggle-ready + FIXES)
# ============================================================
import os
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from io import BytesIO
import base64
from datetime import datetime

%matplotlib inline

print("Starting pipeline: mBERT → 3 Poolings → SVM")

# -----------------------------
# CONFIG
# -----------------------------
MODEL_NAME = "bert-base-multilingual-cased"
MAX_LEN = 64
BATCH_SIZE = 16        # safer for Kaggle GPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TRAIN_PATH = "/kaggle/input/your-dataset/training_clean.csv"
TEST_PATH  = "/kaggle/input/your-dataset/testing_clean.csv"

OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)

print("Device:", DEVICE)
print("Model name:", MODEL_NAME)

# -----------------------------
# LOAD MODEL (FIXED)
# No trust_remote_code to avoid "additional_chat_templates" lookup
# -----------------------------
from transformers import BertTokenizerFast, BertModel

print("\nLoading tokenizer and model...")

try:
    tokenizer = BertTokenizerFast.from_pretrained(
        MODEL_NAME,
        local_files_only=False   # still allowed, but does NOT trigger chat templates
    )

    model = BertModel.from_pretrained(
        MODEL_NAME,
        local_files_only=False
    )

    model.to(DEVICE)
    model.eval()
    print("Model loaded successfully with BertTokenizerFast + BertModel.")

except Exception as e:
    print("❌ Direct load failed:", e)
    print("If Kaggle blocks it, download mBERT manually into /kaggle/input/")
    raise

# -----------------------------
# POOLING FUNCTIONS
# -----------------------------
def mean_pooling(hidden, mask):
    mask = mask.unsqueeze(-1).expand(hidden.size()).float()
    masked = hidden * mask
    summed = masked.sum(dim=1)
    count = mask.sum(dim=1).clamp(min=1e-9)
    return summed / count

def max_pooling(hidden, mask):
    mask = mask.unsqueeze(-1).expand(hidden.size()).float()
    hidden = hidden.masked_fill(mask == 0, -1e9)
    return hidden.max(dim=1).values

def cls_pooling(hidden):
    return hidden[:, 0, :]

# -----------------------------
# EMBEDDING GENERATION
# -----------------------------
def generate_embeddings(texts, batch_size=BATCH_SIZE, max_len=MAX_LEN):
    mean_list, cls_list, multi_list = [], [], []
    model.eval()

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Batches"):
            batch = texts[i:i+batch_size]

            enc = tokenizer(
                batch,
                padding="max_length",
                truncation=True,
                max_length=max_len,
                return_tensors="pt"
            ).to(DEVICE)

            out = model(**enc)
            hidden = out.last_hidden_state
            mask = enc["attention_mask"]

            m = mean_pooling(hidden, mask)
            mx = max_pooling(hidden, mask)
            c = cls_pooling(hidden)
            multi = torch.cat([m, mx, c], dim=1)

            mean_list.append(m.cpu().numpy())
            cls_list.append(c.cpu().numpy())
            multi_list.append(multi.cpu().numpy())

    return (
        np.vstack(mean_list),
        np.vstack(cls_list),
        np.vstack(multi_list)
    )

# -----------------------------
# LOAD CLEANED CSV
# -----------------------------
df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

train_texts = df_train["text"].astype(str).tolist()
test_texts  = df_test["text"].astype(str).tolist()
y_train = df_train["label"].astype(str).values
y_test  = df_test["label"].astype(str).values

print(f"Train={len(train_texts)}  Test={len(test_texts)}")

# -----------------------------
# GENERATE EMBEDDINGS
# -----------------------------
print("\nGenerating TRAIN embeddings...")
train_mean, train_cls, train_multi = generate_embeddings(train_texts)

print("\nGenerating TEST embeddings...")
test_mean, test_cls, test_multi = generate_embeddings(test_texts)

# -----------------------------
# SAVE
# -----------------------------
np.save(f"{OUT_DIR}/train_mean.npy", train_mean)
np.save(f"{OUT_DIR}/test_mean.npy", test_mean)
np.save(f"{OUT_DIR}/train_cls.npy", train_cls)
np.save(f"{OUT_DIR}/test_cls.npy", test_cls)
np.save(f"{OUT_DIR}/train_multi.npy", train_multi)
np.save(f"{OUT_DIR}/test_multi.npy", test_multi)

# -----------------------------
# CONFUSION MATRIX → BASE64
# -----------------------------
def cm_to_base64(cm, labels, title):
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    buf = BytesIO()
    plt.savefig(buf, format="png", bbox_inches="tight")
    buf.seek(0)
    encoded = base64.b64encode(buf.read()).decode()
    plt.close()
    return encoded

# -----------------------------
# TRAIN + EVALUATE SVM
# -----------------------------
def evaluate_svm(Xtr, Xts):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(Xtr)
    Xts = scaler.transform(Xts)

    clf = LinearSVC(C=1.0, max_iter=30000, dual=False)
    clf.fit(Xtr, y_train)
    pred = clf.predict(Xts)

    acc = accuracy_score(y_test, pred)
    f1  = f1_score(y_test, pred, average="macro")
    rep = classification_report(y_test, pred)
    labels = clf.classes_

    cm = confusion_matrix(y_test, pred, labels=labels)
    cm_b64 = cm_to_base64(cm, labels, "Confusion Matrix")

    return acc, f1, rep, cm_b64, labels

# -----------------------------
# RUN ALL 3
# -----------------------------
results = {}

sets = {
    "Mean Pooling": (train_mean, test_mean),
    "CLS Pooling":  (train_cls, test_cls),
    "Mean+Max+CLS": (train_multi, test_multi)
}

for k,(Xtr, Xts) in sets.items():
    print("Evaluating:", k)
    acc, f1, rep, cm_b64, labels = evaluate_svm(Xtr, Xts)
    results[k] = dict(acc=acc, f1=f1, rep=rep, cm=cm_b64)

# -----------------------------
# HTML REPORT
# -----------------------------
html = f"""
<html>
<head><title>BERT + SVM Report</title></head>
<body>
<h1>BERT (mBERT) → Poolings → SVM</h1>
<p>Generated: {datetime.now()}</p>
<p>Device: {DEVICE}</p>
<hr>
"""

for name, r in results.items():
    html += f"""
    <h2>{name}</h2>
    <p><b>Accuracy:</b> {r['acc']:.4f} |
       <b>Macro F1:</b> {r['f1']:.4f}</p>
    <pre>{r['rep']}</pre>
    <img width='600' src='data:image/png;base64,{r['cm']}'/>
    <hr>
    """

html += "</body></html>"

with open(f"{OUT_DIR}/bert_report.html","w",encoding="utf-8") as f:
    f.write(html)

print("\n✅ DONE — Report saved to /kaggle/working/bert_report.html")


Starting pipeline: mBERT → 3 Poolings → SVM
Device: cuda
Model name: bert-base-multilingual-cased

Loading tokenizer and model...
Model loaded successfully with BertTokenizerFast + BertModel.
Train=8954  Test=2244

Generating TRAIN embeddings...


Batches: 100%|██████████| 560/560 [00:32<00:00, 17.27it/s]



Generating TEST embeddings...


Batches: 100%|██████████| 141/141 [00:08<00:00, 16.55it/s]


Evaluating: Mean Pooling
Evaluating: CLS Pooling
Evaluating: Mean+Max+CLS

✅ DONE — Report saved to /kaggle/working/bert_report.html
